> **제2회 Level 모의고사** — 교재 594~655쪽
> 문제 594~596쪽 / 정답·해설 632~655쪽
>
> 노트북과 같은 위치에 데이터 파일(csv)을 두고 위에서부터 차례대로 실행하세요.

# 작업형 제1유형 

## 문제 1. (10점)

제공된 데이터(quiz_answers.csv)는 학생들이 각 문제를 풀었을 때의 정답 여부를 기록한 자료이다. 각 문제는 대주제, 소주제로 분류되어 있다. 아래 수행 순서에 따라 데이터를 처리한 뒤, 구한 값을 [제출 형식]에 맞춰 답안을 입력하시오.

**[수행 순서]**
1. 소주제별 정답률을 계산한다. (정답률 = 해당 소주제의 정답 개수 / 해당 소주제의 전체 문제 풀이 수)
2. 소주제들의 정답률 중 3번째로 높은 값을 구하시오. 단, 정답률이 같은 경우 하나로 취급한다.

**제공 데이터 (quiz_answers.csv)**
| 컬럼명 | 설명 |
|---|---|
| major_topic | 대주제 |
| sub_topic | 소주제 |
| student_id | 학생 ID |
| is_correct | 정답 여부 (1: 정답, 0: 오답) |

**[제출 형식]**
- 반올림하여 소수점 셋째 자리까지 작성

### 문제 풀이 방향

한 행이 학생 한 명의 문제 풀이 결과를 나타내므로, 소주제를 기준으로 여러 행을 하나로 묶어서 비율을 구하는 작업이 먼저 필요합니다. 그 다음 정답률 값들 중에서 동률을 하나로 취급한 뒤 3번째로 큰 값을 찾아야 합니다. 그룹별 비율 계산과 중복 제거, 순위 조회까지 세 단계로 나누어 풀어보겠습니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [1]:
import pandas as pd

df = pd.read_csv('quiz_answers.csv')
print(df.head())

  major_topic sub_topic student_id  is_correct
0          수학        대수   S0000_대수           0
1          수학        대수   S0001_대수           1
2          수학        대수   S0002_대수           1
3          수학        대수   S0003_대수           0
4          수학        대수   S0004_대수           1


**셀 2) 소주제별 정답률 계산하기**

In [2]:
acc = df.groupby('sub_topic')['is_correct'].mean().round(3)
print(acc)

sub_topic
기하       0.571
대수       0.625
독해       0.640
문법       0.800
문학       0.680
물리       0.509
화학       0.680
확률과통계    0.723
Name: is_correct, dtype: float64


`groupby('sub_topic')`은 sub_topic 값이 같은 행들을 하나의 묶음으로 모읍니다. 여기에 `['is_correct'].mean()`을 적용하면 각 묶음 안에서 is_correct 값의 평균이 계산됩니다. is_correct는 정답이 1, 오답이 0으로만 되어 있으므로, 이 평균이 곧 정답률과 같은 값이 됩니다. 예를 들어 5개 중 3개가 정답이면 1, 1, 1, 0, 0의 평균은 0.6이고, 이는 정답률 60%와 같은 수치입니다. 정답 개수를 따로 세고 전체 개수로 나누는 두 단계를 거치지 않고, 평균이라는 한 가지 연산으로 정답률이 바로 구해지는 것이 이 코드의 핵심입니다.

> 💡 0과 1로만 이루어진 컬럼의 평균은 곧 비율(퍼센트)이라는 성질은 정답률뿐 아니라 찬성/반대, 성공/실패처럼 이진 값을 다루는 문제에서 자주 재사용됩니다.

**셀 3) 3번째로 높은 정답률 찾기 (동률 처리 포함)**

In [3]:
unique_sorted = sorted(set(acc), reverse=True)
answer = unique_sorted[2]

print(unique_sorted)
print("정답:", answer)

[0.8, 0.723, 0.68, 0.64, 0.625, 0.571, 0.509]
정답: 0.68


문제에서 정답률이 같으면 하나로 취급하라고 했으므로, 먼저 중복된 값을 제거해야 합니다. 위 출력을 보면 화학과 문학이 똑같이 0.68이라는 정답률을 가지고 있는데, 이를 그대로 두면 값이 8개로 세어져서 순위가 실제와 다르게 밀리게 됩니다. `set(acc)`는 파이썬에서 중복을 허용하지 않는 자료형이므로, Series를 집합으로 감싸는 것만으로 중복된 0.68이 자동으로 하나로 합쳐집니다.

`sorted(..., reverse=True)`로 큰 값부터 작은 값 순서로 정렬한 리스트를 만들면, 인덱스 0번이 1등, 1번이 2등, 2번이 3등에 해당합니다. 따라서 `unique_sorted[2]`가 곧 3번째로 높은 정답률입니다.

### 전체 코드

In [4]:
import pandas as pd

df = pd.read_csv('quiz_answers.csv')

acc = df.groupby('sub_topic')['is_correct'].mean().round(3)
unique_sorted = sorted(set(acc), reverse=True)
answer = unique_sorted[2]

print("정답:", answer)

정답: 0.68


### 정리

이 문제의 핵심은 두 가지입니다. 하나는 0과 1로 이루어진 컬럼의 평균이 곧 비율이라는 성질을 이용해 정답률 계산을 한 줄로 끝낸 것이고, 다른 하나는 set()으로 동률을 하나로 합친 뒤 순위를 매긴 것입니다. 동률 처리를 빠뜨리면 답이 실제보다 낮은 순위의 값으로 잘못 나올 수 있으므로, 값이 같으면 하나로 취급하라는 지시가 있을 때는 항상 중복 제거 단계를 먼저 거치는 습관이 필요합니다.

---

## 문제 2. (10점)

제공된 데이터(cafe_sales.csv)는 2024년 4월부터 2025년 12월까지의 카페 메뉴 판매 이력입니다. 소문항을 각각 구한 다음 [제출 형식]에 맞춰 답안을 입력하시오.

**문제 2-1** 연월별 총매출액을 계산하여 큰 순서대로 정렬했을 때, 2번째로 큰 매출액을 입력하시오.

**문제 2-2** 연월별 총매출액이 4번째로 큰 연-월을 찾으시오. 해당 연-월의 데이터에서 카테고리별 매출 합계를 계산하고, 그중 최댓값을 입력하시오.

**제공 데이터 (cafe_sales.csv)**
| 컬럼명 | 설명 |
|---|---|
| order_date | 주문 날짜 (YYYY-MM-DD) |
| category | 음료/디저트 종류 |
| item | 메뉴명 |
| price | 판매 금액 |

**[제출 형식]**
- 정수로 작성

### 문제 풀이 방향

원본 데이터는 하루 단위(order_date)로 기록되어 있지만 문제는 월 단위 매출을 요구하므로, 먼저 날짜에서 연-월 정보만 뽑아내는 작업이 필요합니다. 그 다음 2-1은 연월별 매출을 집계해서 등수를 매기는 문제이고, 2-2는 특정 등수의 연월을 먼저 찾은 뒤 그 안에서 다시 카테고리 기준으로 한 번 더 집계하는, 두 단계로 이어지는 문제입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 연월(ym) 컬럼 만들기**

In [5]:
import pandas as pd

df = pd.read_csv('cafe_sales.csv')
df['ym'] = df['order_date'].str[:7]

print(df[['order_date', 'ym']].head())

   order_date       ym
0  2024-04-16  2024-04
1  2024-04-19  2024-04
2  2024-04-20  2024-04
3  2024-04-24  2024-04
4  2024-04-21  2024-04


`order_date`는 `'2024-04-16'` 형태의 문자열입니다. `.str[:7]`은 문자열 컬럼 전체에서 앞 7글자만 잘라내는 함수입니다. 날짜 형식이 `YYYY-MM-DD`로 고정되어 있으므로, 앞의 7글자(`2024-04`)만 잘라내면 연도 4자리와 하이픈, 월 2자리가 정확히 남아 연-월 정보가 됩니다. 만약 일(day) 단위 그대로 그룹을 나누면 같은 달이어도 날짜가 다른 주문은 전부 다른 그룹으로 취급되어 월별 합계를 구할 수 없으므로, 이렇게 일자 정보를 버리고 연-월까지만 남긴 새로운 기준값을 만드는 것입니다.

> 💡 날짜 형식이 항상 일정하지 않을 수도 있다면, 문자열 슬라이싱보다 `pd.to_datetime()`으로 날짜 타입으로 변환한 뒤 `.dt.to_period('M')`을 쓰는 방법이 더 안전합니다. pandas가 다양한 날짜 표기를 자동으로 인식해주기 때문입니다.

**셀 2) 연월별 매출 합계를 구하고 정렬하기**

In [7]:
ym_sum = df.groupby('ym')['price'].sum().sort_values(ascending=False)
print(ym_sum)

ym
2024-10    2027500
2024-11    1930000
2025-03    1906500
2025-02    1868000
2025-04    1864000
2025-08    1847500
2024-07    1691500
2025-12    1621000
2025-10    1592000
2025-05    1496000
2025-01    1405500
2024-05    1393000
2024-04    1341000
2024-09    1258500
2024-08    1222500
2025-09    1192500
2024-12    1171000
2024-06     968000
2025-07     958500
2025-11     888500
2025-06     842000
Name: price, dtype: int64


`groupby('ym')['price'].sum()`으로 연월별 매출 합계를 구한 뒤, 바로 `sort_values(ascending=False)`로 큰 값부터 정렬해두었습니다. 이렇게 미리 정렬까지 해두면 등수를 조회하는 다음 단계가 단순해집니다.

**셀 3) 2번째로 큰 매출액 찾기 (문제 2-1 정답)**

In [8]:
answer_2_1 = ym_sum.iloc[1]
print("문제 2-1 정답:", answer_2_1)

문제 2-1 정답: 1930000


정렬된 Series에서는 `.iloc[0]`이 1등, `.iloc[1]`이 2등에 해당하므로, 2번째로 큰 매출액은 `.iloc[1]`로 바로 꺼낼 수 있습니다.

**셀 4) 4번째로 큰 연월을 찾고, 그 달의 카테고리별 매출 최댓값 구하기 (문제 2-2 정답)**

In [10]:
fourth_ym = ym_sum.index[3]
print("4번째로 큰 연월:", fourth_ym)

sub = df[df['ym'] == fourth_ym]
answer_2_2 = sub.groupby('category')['price'].sum().max()

print("문제 2-2 정답:", answer_2_2)

4번째로 큰 연월: 2025-02
문제 2-2 정답: 596500


`ym_sum.index[3]`은 정렬된 Series에서 4번째 행의 인덱스, 즉 연월 이름을 가져옵니다. 값이 아니라 이름(인덱스)이 필요하므로 `.iloc` 대신 `.index`를 사용했습니다.

그 다음 `df[df['ym'] == fourth_ym]`으로 그 연월에 해당하는 행만 걸러내고, 그 부분 데이터 안에서 다시 `groupby('category')['price'].sum()`으로 카테고리별 합계를 구했습니다. 이처럼 먼저 조건으로 범위를 좁힌 뒤 그 안에서 다른 기준으로 다시 집계하는 방식은, 특정 달 안에서 혹은 특정 부서 안에서와 같이 조건이 겹으로 걸린 문제에서 자주 사용됩니다.

### 전체 코드

In [11]:
import pandas as pd

df = pd.read_csv('cafe_sales.csv')
df['ym'] = df['order_date'].str[:7]

ym_sum = df.groupby('ym')['price'].sum().sort_values(ascending=False)
answer_2_1 = ym_sum.iloc[1]
print("문제 2-1 정답:", answer_2_1)

fourth_ym = ym_sum.index[3]
sub = df[df['ym'] == fourth_ym]
answer_2_2 = sub.groupby('category')['price'].sum().max()
print("문제 2-2 정답:", answer_2_2)

문제 2-1 정답: 1930000
문제 2-2 정답: 596500


### 정리

이 문제는 날짜에서 필요한 단위(연-월)만 추출하는 전처리, 그룹 집계 후 정렬, 그리고 조건으로 범위를 좁힌 뒤 다시 집계하는 2단계 그룹화까지 한 번에 연습할 수 있는 문제였습니다. 특히 2-2처럼 값을 찾는 것과 그 값에 해당하는 이름(연월)을 찾는 것이 다른 작업이라는 점을 구분하는 것이 중요합니다. iloc은 정렬된 순서상의 위치(값)를, index는 그 위치에 해당하는 이름을 가져온다는 차이를 기억해두면 비슷한 문제에서 헷갈리지 않습니다.

---

## 문제 3. (10점)

제공된 데이터(air_quality.csv)는 대기 중 오염물질 측정 데이터입니다. 아래 수행 순서에 따라 데이터를 처리한 후, 구한 값을 [제출 형식]에 맞춰 답안을 입력하시오.

**[수행 순서]**
1. 주어진 데이터에서 `co`와 `nmhc` 컬럼을 각각 Min-Max 스케일링하시오.
2. 스케일링된 `co`, `nmhc` 컬럼의 표준편차를 각각 구하시오.
3. `co` 컬럼의 표준편차에서 `nmhc` 컬럼의 표준편차를 뺀 값을 구하시오.

```
Min-Max 스케일링 = (X - min_X) / (max_X - min_X)
```

**제공 데이터 (air_quality.csv)**
| 컬럼명 | 설명 |
|---|---|
| co | 일산화탄소 농도 |
| nmhc | 비메탄 탄화수소 농도 |
| o3 | 오존 농도 |
| no2 | 이산화질소 농도 |

**[제출 형식]**
- 반올림하여 소수점 셋째 자리까지 작성

### 문제 풀이 방향

co와 nmhc는 원래 값의 범위가 서로 크게 다른 컬럼입니다. 이 상태로 표준편차를 비교하면 원래 숫자가 큰 컬럼이 항상 더 퍼져 보이는 착시가 생기므로, 두 컬럼을 먼저 0~1 사이의 같은 범위로 맞춘 뒤에 표준편차를 비교해야 공정한 비교가 됩니다. 스케일링, 표준편차 계산, 두 값의 차이라는 세 단계로 이어지는 문제입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [12]:
import pandas as pd

df = pd.read_csv('air_quality.csv')
print(df.head())

      co    nmhc     o3    no2
0  1.546   92.94  42.39  50.98
1  0.403  204.06  62.55  27.08
2  1.683  195.67  55.15  29.80
3  2.421   15.26  76.48  22.22
4  1.777   82.56  19.39  45.26


**셀 2) co, nmhc를 각각 Min-Max 스케일링하기**

In [13]:
co_scaled = (df['co'] - df['co'].min()) / (df['co'].max() - df['co'].min())
nmhc_scaled = (df['nmhc'] - df['nmhc'].min()) / (df['nmhc'].max() - df['nmhc'].min())

print(co_scaled.head())
print(nmhc_scaled.head())

0    0.099491
1    0.023931
2    0.108548
3    0.157335
4    0.114762
Name: co, dtype: float64
0    0.214143
1    0.510660
2    0.488272
3    0.006858
4    0.186444
Name: nmhc, dtype: float64


`df['co'].min()`과 `df['co'].max()`로 컬럼의 최솟값과 최댓값을 구하고, 공식 그대로 컬럼 전체에 대입했습니다. 분자인 `X - min_X`는 그 값이 최솟값보다 얼마나 큰지를 나타내고, 분모인 `max_X - min_X`는 컬럼 전체 값이 퍼져 있는 폭을 나타냅니다. 이 둘을 나누면 전체 폭 중에서 그 값이 어느 위치(비율)에 있는지가 계산됩니다. 그 결과 최솟값이었던 데이터는 0이 되고 최댓값이었던 데이터는 1이 되며, 나머지 값은 그 사이의 비율로 표현됩니다. 이렇게 하면 원래 값이 얼마였든 상관없이 두 컬럼이 똑같이 0에서 1 사이로 맞춰집니다.

**셀 3) 스케일링된 값의 표준편차 구하고 차이 계산하기 (정답)**

In [14]:
co_std = co_scaled.std()
nmhc_std = nmhc_scaled.std()

answer = round(co_std - nmhc_std, 3)

print("co 표준편차:", co_std)
print("nmhc 표준편차:", nmhc_std)
print("정답:", answer)

co 표준편차: 0.11314582089057235
nmhc 표준편차: 0.17037086762974993
정답: -0.057


`.std()`는 표준편차를 구해주는 함수로, 데이터가 평균에서 얼마나 떨어져 있는지를 나타냅니다. pandas의 `.std()`는 기본적으로 표본표준편차 방식(n이 아니라 n-1로 나누는 방식)을 사용합니다. 답이 음수로 나온 것은 스케일링 후에도 nmhc 값들이 co 값들보다 0과 1 사이에서 더 넓게 퍼져 있다는 뜻입니다. 스케일링은 값의 범위만 0에서 1로 맞춰줄 뿐, 그 안에서 데이터가 얼마나 고르게 분포하는지는 원래 데이터의 특성을 그대로 반영하기 때문입니다.

> 💡 서로 다른 단위나 범위를 가진 컬럼을 비교해야 할 때는 표준편차나 분산을 그대로 비교하기 전에 스케일링(Min-Max 또는 표준화)을 먼저 거치는 것이 일반적인 접근입니다.

### 전체 코드

In [15]:
import pandas as pd

df = pd.read_csv('air_quality.csv')

co_scaled = (df['co'] - df['co'].min()) / (df['co'].max() - df['co'].min())
nmhc_scaled = (df['nmhc'] - df['nmhc'].min()) / (df['nmhc'].max() - df['nmhc'].min())

answer = round(co_scaled.std() - nmhc_scaled.std(), 3)
print("정답:", answer)

정답: -0.057


### 정리

이 문제는 스케일링 자체의 계산은 어렵지 않지만, 왜 스케일링을 먼저 해야 하는지를 이해하는 것이 핵심이었습니다. 서로 범위가 다른 두 컬럼을 그대로 비교하면 원래 값의 크기 차이가 결과를 왜곡시키므로, 비교하기 전에 같은 기준(0~1)으로 맞추는 절차가 필요합니다. 스케일링, 표준편차, 뺄셈이라는 세 함수만으로 문제가 끝났다는 점에서, 계산 자체보다 순서와 이유를 이해하는 것이 더 중요한 문제라고 볼 수 있습니다.

---

## 이번 회차에서 사용한 핵심 함수 정리
| 상황 | 함수 | 핵심 이유 |
|---|---|---|
| 0/1 값으로 비율(정답률) 구하기 | `groupby().mean()` | 이진 값의 평균이 곧 비율이 된다는 성질 이용 |
| 값이 같으면 하나로 취급 (동률 처리) | `set()` | 중복을 허용하지 않는 자료형이라 자동으로 중복 제거 |
| 날짜 문자열에서 연-월만 추출 | `.str[:7]` | 고정된 날짜 형식에서 필요한 구간만 슬라이싱 |
| 그룹 합계 후 등수 조회 | `groupby().sum().sort_values()` + `.iloc[n-1]` | 정렬 후 인덱스 위치로 등수를 바로 조회 |
| 값이 아니라 이름(인덱스)이 필요할 때 | `.index[n]` | iloc은 값의 위치, index는 그 위치의 이름 |
| 조건으로 범위를 좁힌 뒤 다시 집계 | `df[조건]` 후 `groupby().sum()` | 조건이 겹으로 걸린 문제에서 2단계로 집계 |
| 서로 다른 범위의 값을 비교 가능하게 맞추기 | `(컬럼 - 컬럼.min()) / (컬럼.max() - 컬럼.min())` | Min-Max 스케일링으로 0~1 범위로 통일 |
| 표준편차 계산 | `.std()` | 기본값이 표본표준편차(n-1)라는 점 유의 |

---

# 작업형 제2유형

## 문제

제공된 학습용 데이터(train.csv)는 전동킥보드 공유 서비스의 시간대별 대여 이력 데이터이다. 학습용 데이터를 활용하여 시간대별 대여량(rental_count)을 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 대여량을 예측하시오. 예측 결과는 아래의 [유의사항]과 [제출 형식]을 준수하여, CSV 파일(result.csv)을 생성하는 코드를 제출하시오.

**[유의사항]**
- ① 예측 결과는 RMSE(Root Mean Squared Error) 평가지표에 따라 평가함
- ② 타겟 변수(rental_count)는 항상 0 이상의 정수이다

**[제출 형식]**
- ① CSV 파일명: result.csv
- ② 예측 대여량 컬럼명: pred
- ③ 제출 컬럼 개수: pred 컬럼 1개
- ④ 평가용 데이터 개수와 예측 결과 데이터 개수 일치: 1,476개

**[제공 데이터]**
- train.csv: 학습용 데이터, 약 3,196개
- test.csv: 평가용 데이터, 약 1,476개 (rental_count 컬럼 미제공)

| 컬럼명 | 설명 |
|---|---|
| id | 기록 ID |
| date | 날짜 (YYYY-MM-DD) |
| hour | 시간대 (0~23) |
| day_of_week | 요일 (0: 월요일 ~ 6: 일요일) |
| season | 계절 (봄/여름/가을/겨울) |
| temp | 기온(℃) |
| humidity | 습도(%) |
| windspeed | 풍속 |
| precipitation | 강수량(mm) |
| is_holiday | 휴일 여부 (0/1) |
| rental_count | (train에만 존재) 해당 시간대의 대여량 |

### 문제 풀이 방향

이 문제는 분류가 아니라 회귀 문제입니다. 예측 대상(rental_count)이 0, 1, 2 같은 범주가 아니라 연속적인 수치이기 때문에, 분류 모델이 아니라 회귀 모델을 사용해야 하고 평가지표도 F1 같은 분류 지표가 아니라 RMSE라는 회귀 지표를 사용합니다. 전체적인 흐름은 1회(분류)와 크게 다르지 않습니다. 데이터를 불러와 확인하고, 날짜처럼 그 자체로는 모델에 넣기 어려운 컬럼을 가공하고, 범주형 변수를 인코딩한 뒤, 검증 데이터로 성능을 가늠해보고, 마지막에 전체 데이터로 재학습해서 예측 결과를 제출하는 순서로 진행합니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기 + 구조 확인**

In [1]:
import pandas as pd

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(train.shape, test.shape)
train.head()

(3196, 11) (1476, 10)


,id,date,hour,day_of_week,season,temp,humidity,windspeed,precipitation,is_holiday,rental_count
0,3439,2023-07-22,6,5,여름,27.1,64.2,12.7,0.0,1,30
1,2460,2023-06-11,11,6,여름,26.7,60.3,9.3,0.0,1,27
2,3548,2023-07-26,19,2,여름,24.3,39.7,13.2,0.0,0,59
3,4328,2023-08-28,7,0,여름,30.1,43.8,10.5,0.0,0,62
4,3172,2023-07-11,3,1,여름,24.1,50.0,9.9,0.0,0,57


train은 11개 컬럼(rental_count 포함), test는 10개 컬럼(rental_count 없음)입니다.

In [2]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3196 entries, 0 to 3195
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             3196 non-null   int64  
 1   date           3196 non-null   object 
 2   hour           3196 non-null   int64  
 3   day_of_week    3196 non-null   int64  
 4   season         3196 non-null   object 
 5   temp           3196 non-null   float64
 6   humidity       3196 non-null   float64
 7   windspeed      3196 non-null   float64
 8   precipitation  3196 non-null   float64
 9   is_holiday     3196 non-null   int64  
 10  rental_count   3196 non-null   int64  
dtypes: float64(4), int64(5), object(2)
memory usage: 274.8+ KB


date 컬럼은 `'2023-07-22'`처럼 문자열로 되어 있어서 이 상태 그대로는 모델이 숫자로 이해할 수 없습니다. season도 마찬가지로 문자열입니다. 이 두 컬럼을 어떻게 처리할지가 이번 문제의 전처리 핵심입니다.

**셀 2) 결측치 및 타겟 분포 확인**

In [3]:
print(train.isna().sum().sum(), test.isna().sum().sum())
print(train['rental_count'].describe())

0 0
count    3196.000000
mean       42.850125
std        15.131135
min         0.000000
25%        32.000000
50%        41.000000
75%        53.000000
max        91.000000
Name: rental_count, dtype: float64


`info()`에서 확인할 수 있었듯이 이번 데이터에는 결측치가 없습니다. 

`rental_count.describe()`로 타겟 변수의 분포도 함께 살펴봤습니다. 최솟값이 0이고 음수는 없다는 것을 확인할 수 있는데, 이는 나중에 모델의 예측값을 후처리할 때 참고할 수 있는 정보입니다. 대여량은 현실적으로 음수가 나올 수 없으므로, 만약 모델이 음수를 예측한다면 0으로 바꿔주는 보정이 필요할 수 있습니다.

**셀 3) 날짜 컬럼에서 파생변수(month) 만들기**

In [4]:
train['month'] = pd.to_datetime(train['date']).dt.month
test['month'] = pd.to_datetime(test['date']).dt.month

print(train[['date', 'month']].head())

         date  month
0  2023-07-22      7
1  2023-06-11      6
2  2023-07-26      7
3  2023-08-28      8
4  2023-07-11      7


`pd.to_datetime()`은 문자열로 된 날짜를 pandas가 진짜 날짜로 인식할 수 있는 타입으로 바꿔줍니다. 날짜 타입으로 바꾸고 나면 `.dt.month`, `.dt.day`, `.dt.dayofweek`처럼 연/월/일/요일 등 원하는 정보를 바로 꺼낼 수 있습니다. 이 문제에서는 이미 day_of_week 컬럼이 제공되어 있으므로, 추가로 month(월)라는 정보를 새로 뽑아냈습니다. 대여량은 계절뿐 아니라 월별로도 패턴이 있을 수 있으므로(예: 같은 여름이라도 6월과 8월의 날씨 체감이 다름), 이렇게 원본 날짜에서 유용한 정보를 추출해 새로운 컬럼(파생변수)으로 만들어주는 작업은 회귀 문제에서 흔히 쓰이는 전처리입니다. date 컬럼 자체는 문자열이라 이후 모델 학습에서는 제외할 예정입니다.

> 💡 날짜 데이터가 주어지면 `pd.to_datetime()`으로 변환한 뒤 연/월/일/요일/주말여부 등 여러 파생변수를 만들어보는 것이 회귀·분류 문제 모두에서 유용한 시도입니다. 날짜 문자열 그 자체보다 이렇게 분해한 정보가 모델이 패턴을 학습하기에 더 유리합니다.

**셀 4) 범주형 변수(season) 인코딩**

In [5]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(pd.concat([train['season'], test['season']]))
train['season'] = le.transform(train['season'])
test['season'] = le.transform(test['season'])

print(train['season'].head())

0    2
1    2
2    2
3    2
4    2
Name: season, dtype: int64


season은 봄/여름/가을/겨울이라는 문자열이므로 숫자로 바꿔야 합니다. 모의고사 1회(분류 문제)에서와 동일하게 `LabelEncoder`를 사용했고, train과 test를 합쳐서 fit했습니다. 이 데이터의 처음 5개 행이 모두 여름철에 몰려 있어서 값이 전부 2로 보이지만, 실제로는 계절마다 0~3 사이의 서로 다른 값이 부여되어 있습니다.

**셀 5) 학습용/검증용 데이터 분리**

In [6]:
from sklearn.model_selection import train_test_split

drop_cols = ['id', 'date']
X = train.drop(columns=drop_cols + ['rental_count'])
y = train['rental_count']
X_test = test.drop(columns=drop_cols)

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_tr.shape, X_val.shape)

(2556, 9) (640, 9)


`id`는 단순 식별자이므로 제외하고, `date`는 이미 month와 day_of_week라는 형태로 필요한 정보를 뽑아냈으므로 문자열 원본은 더 이상 필요하지 않아 함께 제외했습니다. 회귀 문제에서는 `train_test_split`에 분류 문제 때 사용했던 `stratify` 옵션을 사용하지 않습니다. stratify는 클래스 비율을 맞추기 위한 옵션인데, rental_count는 범주가 아니라 연속적인 숫자이므로 클래스라는 개념 자체가 없기 때문입니다.

**셀 6) 모델 학습 및 검증 성능 확인**

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)

val_pred = model.predict(X_val)
rmse = mean_squared_error(y_val, val_pred) ** 0.5
print("검증 RMSE:", rmse)

검증 RMSE: 8.72218821096518


분류에서 `RandomForestClassifier`를 사용했던 것처럼, 회귀에서는 `RandomForestRegressor`를 사용합니다. 이름 그대로 분류(Classifier)냐 회귀(Regressor)냐의 차이이며, 사용법은 거의 동일합니다.

`mean_squared_error`는 실제값과 예측값 차이를 제곱해서 평균 낸 값(MSE)을 계산해주는 함수입니다. 문제에서 요구한 지표는 RMSE인데, RMSE는 MSE에 제곱근을 씌운 값입니다. `mean_squared_error(...) ** 0.5`처럼 제곱근을 직접 씌워서 RMSE를 구했습니다. 제곱근을 씌우는 이유는, MSE는 오차를 제곱했기 때문에 단위가 원래 값의 제곱(대여량의 제곱)이 되어 해석하기 어려운 반면, 제곱근을 씌우면 다시 원래 단위(대여 건수)로 돌아와서 오차의 크기를 직관적으로 이해할 수 있기 때문입니다. 여기서 RMSE가 약 8.7이라는 것은, 평균적으로 예측이 실제 대여량과 8~9건 정도 차이 난다는 의미로 해석할 수 있습니다. 타겟값의 평균이 약 43, 표준편차가 약 15임을 고려하면 상당히 준수한 수준입니다.

> 💡 RMSE는 값이 작을수록 좋은 지표입니다. F1이나 정확도(Accuracy)처럼 값이 클수록 좋은 지표와 방향이 반대라는 점을 기억해두면, 검증 점수를 비교하며 모델을 개선할 때 혼동을 줄일 수 있습니다.

**셀 7) 전체 학습 데이터로 재학습 후 test 예측**

In [8]:
import numpy as np

model.fit(X, y)
pred = model.predict(X_test)
pred = np.clip(pred, 0, None)

print(pred[:10])

[46.56       41.12       40.47333333 31.26666667 65.82666667 51.7
 29.08       59.66333333 33.03666667 41.69333333]


검증까지 마친 뒤에는 train 데이터 전체를 사용해 다시 학습시킵니다. 여기서 회귀 문제이기 때문에 추가로 필요한 처리가 하나 있습니다. `np.clip(pred, 0, None)`은 예측값 중 혹시라도 0보다 작은(음수) 값이 있다면 전부 0으로 바꿔주는 함수입니다. 셀 2에서 확인했듯이 대여량은 현실적으로 음수가 될 수 없는데, 회귀 모델은 이런 제약을 스스로 알지 못하기 때문에 이론적으로는 음수를 예측할 수도 있습니다. 이렇게 도메인 지식(대여량은 0 이상이어야 한다는 사실)을 바탕으로 예측값을 보정해주는 것은 회귀 문제에서 자주 쓰이는 후처리입니다.

**셀 8) 제출 파일 생성**

In [9]:
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)

print(result.shape)
print(result.head())

(1476, 1)
        pred
0  46.560000
1  41.120000
2  40.473333
3  31.266667
4  65.826667


행 개수가 요구된 1,476개와 정확히 일치하는지 확인했습니다. 예측값이 정수가 아니라 소수로 나온 것은 회귀 모델의 예측이 연속적인 값이기 때문이며, 특별히 정수로 반올림하라는 지시가 없다면 소수 형태로 제출해도 무방합니다.

### 전체 코드

In [10]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

# 날짜에서 월 파생변수 생성
train['month'] = pd.to_datetime(train['date']).dt.month
test['month'] = pd.to_datetime(test['date']).dt.month

# 범주형 인코딩
le = LabelEncoder()
le.fit(pd.concat([train['season'], test['season']]))
train['season'] = le.transform(train['season'])
test['season'] = le.transform(test['season'])

drop_cols = ['id', 'date']
X = train.drop(columns=drop_cols + ['rental_count'])
y = train['rental_count']
X_test = test.drop(columns=drop_cols)

# 검증용 분리 후 성능 확인
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=300, random_state=42)
model.fit(X_tr, y_tr)
val_pred = model.predict(X_val)
print("검증 RMSE:", mean_squared_error(y_val, val_pred) ** 0.5)

# 전체 데이터로 재학습 후 test 예측 (음수 보정 포함)
model.fit(X, y)
pred = model.predict(X_test)
pred = np.clip(pred, 0, None)

# 제출 파일 생성
result = pd.DataFrame({'pred': pred})
result.to_csv('result.csv', index=False)
print(result.shape)

검증 RMSE: 8.72218821096518
(1476, 1)


### 정리

이 문제는 1회(분류)와 전체 골격은 같지만, 세부적으로 다른 지점들이 있었습니다. 평가지표가 F1-macro에서 RMSE로 바뀌면서 점수가 낮을수록 좋다는 방향이 반대가 되었고, 모델도 Classifier에서 Regressor로 바뀌었으며, train_test_split에서 stratify를 사용하지 않았고, 예측 후에는 음수를 0으로 보정하는 회귀 특유의 후처리가 추가되었습니다. 또한 날짜 문자열에서 month라는 파생변수를 뽑아내는 과정도 새롭게 다뤘습니다. 분류와 회귀는 비슷한 흐름을 공유하지만 이런 세부 차이를 놓치지 않는 것이 중요합니다.

이번 모델의 검증 RMSE는 약 8.7 수준이었습니다. 시간이 남는다면 `n_estimators`(트리 개수)를 늘려보거나, `GradientBoostingRegressor` 같은 다른 모델을 시도해보거나, hour나 temp 같은 변수들을 조합한 새로운 파생변수를 만들어보는 것도 점수를 더 개선하는 방법이 될 수 있습니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 문자열 날짜를 다룰 수 있는 타입으로 변환 | `pd.to_datetime()` | 변환 후 `.dt.month`, `.dt.dayofweek` 등으로 정보 추출 가능 |
| 날짜에서 유용한 정보를 새 컬럼으로 뽑아내기 | 파생변수 생성 (`.dt.month` 등) | 원본 문자열보다 모델이 학습하기 좋은 형태로 가공 |
| 회귀 문제에서의 데이터 분리 | `train_test_split` (stratify 미사용) | 연속값에는 클래스 비율이라는 개념이 없음 |
| 분류 모델과 짝을 이루는 회귀 모델 | `RandomForestRegressor` | 분류(Classifier)와 이름만 다르고 사용법은 유사 |
| 회귀 평가지표 계산 | `mean_squared_error(y, pred) ** 0.5` | MSE에 제곱근을 씌우면 원래 단위로 해석 가능한 RMSE가 됨 |
| 현실적으로 불가능한 예측값 보정 | `np.clip(pred, 0, None)` | 모델은 도메인 제약(음수 불가 등)을 스스로 알지 못하므로 후처리 필요 |

---

# 작업형 제3유형

## 문제 1. (10점)

제공된 데이터(membership_survey.csv)는 온라인 쇼핑몰 고객의 회원등급과 선호 구매 카테고리에 대한 설문 자료이다. 회원등급(membership_tier)과 선호 카테고리(preferred_category) 간에 통계적으로 유의한 연관성이 있는지 카이제곱 독립성검정을 수행하고, 아래 소문항에 답하시오.

**제공 데이터 (membership_survey.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 고객 번호 |
| membership_tier | 회원등급 (Bronze/Silver/Gold) |
| preferred_category | 선호 구매 카테고리 (전자제품/의류/식품/뷰티) |

**문제 1-1** membership_tier와 preferred_category의 교차표(관측빈도표)를 생성하시오. Silver 등급이면서 의류를 선호하는 고객 수를 구하시오. (정수로 작성)

**문제 1-2** membership_tier와 preferred_category 간의 카이제곱 독립성검정을 수행하고, 카이제곱 검정통계량을 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 1-3** 위 검정의 p-value를 구하고, 유의수준 0.05에서 귀무가설을 기각하는지 채택하는지 판단하시오.

### 문제 풀이 방향

카이제곱 독립성검정은 "두 범주형 변수가 서로 관련이 있는가"를 확인하는 검정입니다. 귀무가설(H0)은 "두 변수는 서로 독립이다(관련이 없다)"이고, 대립가설(H1)은 "두 변수는 독립이 아니다(관련이 있다)"입니다. 이 검정을 수행하려면 먼저 두 범주형 변수를 행과 열로 교차시킨 표(교차표, contingency table)를 만들어야 하고, 그 표를 `scipy.stats`의 `chi2_contingency()` 함수에 넣으면 검정통계량과 p-value가 한 번에 계산됩니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [1]:
import pandas as pd

df = pd.read_csv('membership_survey.csv')
print(df.shape)
print(df.head())

(620, 3)
   id membership_tier preferred_category
0   1          Silver                 뷰티
1   2          Bronze                 식품
2   3          Silver                 뷰티
3   4            Gold                 뷰티
4   5          Silver                 식품


620명의 고객 데이터이며, membership_tier와 preferred_category 모두 문자열로 된 범주형 변수입니다. 한 행이 고객 한 명을 나타내며, 아직은 등급별로 어떤 카테고리를 얼마나 선호하는지 요약되지 않은 원본(raw) 형태입니다.

**셀 2) 교차표 생성 (문제 1-1 정답)**

In [2]:
ct = pd.crosstab(df['membership_tier'], df['preferred_category'])
print(ct)

print("\n1-1 정답:", ct.loc['Silver', '의류'])

preferred_category  뷰티  식품   의류  전자제품
membership_tier                      
Bronze              36  77  104    59
Gold                34  30   17    47
Silver              45  66   45    60

1-1 정답: 45


`pd.crosstab(행 기준, 열 기준)`은 두 범주형 컬럼을 각각 행과 열로 삼아, 두 조건을 동시에 만족하는 데이터가 몇 건인지 세어 표로 만들어주는 함수입니다. 이 표가 바로 카이제곱 검정에 사용할 관측빈도표(observed frequency table)입니다. 표에서 Silver 행과 의류 열이 만나는 칸의 값이 45이므로, "Silver 등급이면서 의류를 선호하는 고객"은 45명입니다. `.loc['Silver', '의류']`처럼 행 이름과 열 이름을 지정하면 교차표의 특정 칸 값을 바로 조회할 수 있습니다.

**셀 3) 카이제곱 독립성검정 수행**

In [3]:
from scipy.stats import chi2_contingency

chi2, p, dof, expected = chi2_contingency(ct)

print("카이제곱 통계량:", chi2)
print("p-value:", p)
print("자유도:", dof)
print("기대빈도:\n", expected)

카이제곱 통계량: 42.63511303016629
p-value: 1.3773274510197203e-07
자유도: 6
기대빈도:
 [[51.19354839 77.01290323 73.89677419 73.89677419]
 [23.74193548 35.71612903 34.27096774 34.27096774]
 [40.06451613 60.27096774 57.83225806 57.83225806]]


`chi2_contingency()`는 교차표 하나만 넣어주면 카이제곱 검정에 필요한 네 가지 값을 한 번에 계산해서 튜플로 돌려줍니다. 첫 번째가 카이제곱 검정통계량, 두 번째가 p-value, 세 번째가 자유도(degrees of freedom), 네 번째가 기대빈도(expected frequency)입니다.

기대빈도는 "만약 두 변수가 정말로 독립이라면, 각 칸에 이 정도의 값이 나왔을 것이다"라고 이론적으로 계산한 값입니다. 예를 들어 Bronze-뷰티 칸의 실제 관측값은 36이었는데 기대빈도는 약 51.19입니다. 카이제곱 검정은 결국 이 관측빈도와 기대빈도가 서로 얼마나 차이 나는지를 종합해서 하나의 숫자(카이제곱 통계량)로 만든 것입니다. 관측값과 기대값의 차이가 클수록 카이제곱 통계량이 커지고, 이는 "두 변수가 독립이라는 가정하에 기대했던 것과 실제 데이터가 많이 다르다"는 뜻이므로 독립이 아닐(연관이 있을) 가능성이 높아집니다.

> 💡 자유도(dof)는 `(행의 개수-1) × (열의 개수-1)`로 계산됩니다. 이 데이터는 행이 3개(Bronze/Silver/Gold), 열이 4개(뷰티/식품/의류/전자제품)이므로 `(3-1)×(4-1)=6`이 되고, 실제 출력된 자유도 6과 일치하는 것을 확인할 수 있습니다.

**셀 4) 카이제곱 통계량 확인 (문제 1-2 정답)**

In [4]:
print("1-2 정답:", round(chi2, 3))

1-2 정답: 42.635


셀 3에서 이미 계산해둔 `chi2` 값을 반올림하기만 하면 됩니다. 카이제곱 통계량 자체는 값이 클수록 관측빈도와 기대빈도의 차이가 크다는 뜻이지만, 이 값 하나만 보고 "얼마나 커야 유의한 차이인지"를 판단하기는 어렵습니다. 그래서 이 통계량을 표준화된 판단 기준인 p-value로 변환해서 해석하는 것이 다음 소문항의 역할입니다.

**셀 5) p-value 확인 및 기각/채택 판단 (문제 1-3 정답)**

In [5]:
alpha = 0.05
p_rounded = round(p, 3)
print("p-value(반올림):", p_rounded)
print("p-value(원래 값):", p)

if p < alpha:
    conclusion = "귀무가설 기각(두 변수는 독립이 아니다, 즉 연관이 있다)"
else:
    conclusion = "귀무가설 채택(두 변수는 독립이다)"
print("1-3 정답:", conclusion)

p-value(반올림): 0.0
p-value(원래 값): 1.3773274510197203e-07
1-3 정답: 귀무가설 기각(두 변수는 독립이 아니다, 즉 연관이 있다)


p-value는 "귀무가설(두 변수가 독립이다)이 참이라고 가정했을 때, 지금 우리가 관측한 것과 같거나 더 극단적인 결과가 나올 확률"입니다. 이 값이 아주 작다는 것은, "정말로 독립이라면 이런 데이터가 나올 가능성은 거의 없다"는 뜻이므로 독립이라는 가정(귀무가설)을 의심하고 기각하게 됩니다. 반대로 p-value가 충분히 크다면(보통 0.05 이상), 지금의 데이터가 두 변수가 독립이더라도 충분히 나올 수 있는 결과라고 보고 귀무가설을 굳이 기각할 근거가 없다고 판단합니다.

이 데이터의 p-value는 약 0.00000014로 유의수준 0.05보다 훨씬 작으므로, 귀무가설을 기각하고 "회원등급과 선호 카테고리는 독립이 아니다"라고 결론 내립니다. 실제로 셀 2의 교차표를 보면 Gold 등급은 전자제품(47명)을 상대적으로 많이 선호하고 의류(17명)는 적게 선호하는 등, 등급마다 선호 패턴이 확연히 다른 것을 확인할 수 있어 이 결론과 일치합니다.

> 💡 p-value가 0.0000001처럼 아주 작은 값일 때, `round(p, 3)`으로 반올림하면 0.0이 되어버려 원래 값의 크기를 알아보기 어려워집니다. 이런 경우 시험에서는 "소수 n자리까지 반올림"이라는 지시를 그대로 따르되, 필요하다면 지수 표기(예: 1.377e-07)나 "p < 0.001"처럼 부등호로 표현하는 방법도 함께 알아두면 좋습니다. 다만 이 문제처럼 제출 형식이 명시되지 않은 경우, "기각/채택 판단"이 핵심이므로 정확한 판단 근거(p-value가 유의수준보다 작다는 것)를 함께 제시하는 것이 중요합니다.

### 전체 코드

In [6]:
import pandas as pd
from scipy.stats import chi2_contingency

df = pd.read_csv('membership_survey.csv')

# 1-1: 교차표 생성
ct = pd.crosstab(df['membership_tier'], df['preferred_category'])
print("1-1 정답:", ct.loc['Silver', '의류'])

# 카이제곱 독립성검정
chi2, p, dof, expected = chi2_contingency(ct)

# 1-2: 카이제곱 통계량
print("1-2 정답:", round(chi2, 3))

# 1-3: p-value 및 기각/채택 판단
alpha = 0.05
conclusion = "귀무가설 기각(연관이 있다)" if p < alpha else "귀무가설 채택(독립이다)"
print("1-3 정답: p-value =", p, "->", conclusion)

1-1 정답: 45
1-2 정답: 42.635
1-3 정답: p-value = 1.3773274510197203e-07 -> 귀무가설 기각(연관이 있다)


### 정리

카이제곱 독립성검정은 "교차표 만들기 → chi2_contingency로 검정통계량·p-value·자유도·기대빈도 한 번에 계산하기 → p-value와 유의수준을 비교해 기각/채택 판단하기"라는 세 단계로 요약할 수 있습니다. 로지스틱·선형 회귀에서 계수와 p-value를 봤던 것처럼, 이 검정에서도 결국 판단의 기준은 p-value라는 점이 같습니다. 다만 회귀분석은 변수들 사이의 "정도"나 "방향"(계수, 오즈비)까지 알려주는 반면, 카이제곱 검정은 "관련이 있다/없다"라는 것만 알려줄 뿐 그 관련성이 구체적으로 어떤 모양인지는 알려주지 않는다는 차이가 있습니다. 관련성의 구체적인 패턴을 보고 싶다면, 이번 문제의 셀 2처럼 교차표 자체를 직접 눈으로 살펴보는 것이 도움이 됩니다.

---

## 문제 2. (10점)

제공된 데이터(card_delinquency.csv)는 카드사 고객의 인적 특성과 카드 이용 특성에 관한 자료이다. id를 기준으로 train(1 ~ 1200)과 test(1201 ~ 1500) 데이터로 나누어, train 데이터로 로지스틱 회귀분석을 수행하여 고객의 연체 여부(delinquent)를 예측하고, test 데이터로 모형의 성능을 평가하시오.

**제공 데이터 (card_delinquency.csv)**
| 컬럼명 | 설명 |
|---|---|
| id | 고객 번호 |
| age | 나이 |
| income | 연소득 |
| credit_limit | 신용카드 한도 |
| num_late_payments | 최근 1년간 연체 횟수 |
| credit_score | 신용점수 |
| delinquent | (종속변수) 연체 여부 — 0: 정상, 1: 연체 |

**문제 2-1** train 데이터에서 id를 제외한 모든 변수를 독립변수로 사용하여 로지스틱 회귀모형을 적합하시오. num_late_payments가 1 증가할 때의 오즈비(Odds Ratio)를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-2** 위 모형으로 test 데이터의 연체 확률을 예측하고, 확률이 0.3을 초과하면 연체(1)로 분류하시오. 이 분류 결과의 민감도(Sensitivity)를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

**문제 2-3** 위 분류 결과의 특이도(Specificity)를 구하시오. (반올림하여 소수 셋째 자리까지 작성)

### 문제 풀이 방향

이 문제는 로지스틱 회귀모형을 학습(train)과 평가(test)로 나누어 다루는 흐름을 담고 있습니다. train 데이터로 모형을 만들고, 그 모형이 한 번도 보지 못한 test 데이터에 적용해 성능을 평가합니다. 또한 예측 확률을 특정 임계값(0.3)을 기준으로 0 또는 1로 분류한 뒤, 그 분류가 얼마나 정확한지를 민감도와 특이도라는 지표로 평가합니다. 민감도와 특이도는 모두 혼동행렬(confusion matrix)의 네 가지 값(TP, FN, TN, FP)으로부터 계산되므로, 이 네 값을 정확히 세는 것이 이 문제의 핵심입니다.

### 정답 코드 및 해설

**셀 1) 데이터 불러오기**

In [7]:
import pandas as pd

df = pd.read_csv('card_delinquency.csv')
print(df.shape)
df.head()

(1500, 7)


,id,age,income,credit_limit,num_late_payments,credit_score,delinquent
0,1,61,5999.0,3356.0,1,631.0,0
1,2,64,6262.0,2636.0,0,746.0,0
2,3,37,1782.0,3466.0,1,684.0,1
3,4,55,4912.0,2915.0,0,697.0,0
4,5,29,4332.0,2060.0,0,676.0,1


전체 1,500명의 데이터이며, 이 안에 train과 test가 모두 섞여 있습니다. id 값을 기준으로 어느 그룹에 속하는지가 정해집니다.

**셀 2) id 기준 train/test 분할**

In [8]:
train = df[df['id'] <= 1200].reset_index(drop=True)
test = df[df['id'] > 1200].reset_index(drop=True)

print("train:", train.shape, "test:", test.shape)
print("train 연체 비율:", round(train['delinquent'].mean(), 3))
print("test 연체 비율:", round(test['delinquent'].mean(), 3))

train: (1200, 7) test: (300, 7)
train 연체 비율: 0.226
test 연체 비율: 0.18


`df['id'] <= 1200`이라는 조건으로 id가 1200 이하인 행만 골라 train으로, 그 나머지(id가 1200 초과, 즉 1201~1500)를 test로 나눴습니다. `reset_index(drop=True)`는 조건으로 걸러낸 뒤 뒤죽박죽이 된 인덱스를 0부터 다시 순서대로 매겨주는 역할로, 필수는 아니지만 이후 작업에서 인덱스를 깔끔하게 다루기 위해 습관적으로 붙여두면 좋습니다. train의 연체 비율(22.6%)과 test의 연체 비율(18%)이 비슷한 수준인 것도 확인해두면, 두 데이터가 크게 다른 성격을 갖고 있지는 않다는 것을 짐작할 수 있습니다.

**셀 3) train 데이터로 로지스틱 회귀모형 적합**

In [9]:
import statsmodels.api as sm

X_train = train.drop(columns=['id', 'delinquent'])
y_train = train['delinquent']
X_train = sm.add_constant(X_train)

model = sm.Logit(y_train, X_train).fit()
print(model.summary())

Optimization terminated successfully.
         Current function value: 0.492935
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:             delinquent   No. Observations:                 1200
Model:                          Logit   Df Residuals:                     1194
Method:                           MLE   Df Model:                            5
Date:                Thu, 09 Jul 2026   Pseudo R-squ.:                 0.07723
Time:                        13:49:03   Log-Likelihood:                -591.52
converged:                       True   LL-Null:                       -641.03
Covariance Type:            nonrobust   LLR p-value:                 8.515e-20
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.8215      0.725      2.511      0.012       0.400       3.243
age     

주의할 점은 `sm.add_constant()`와 `sm.Logit().fit()`을 test 데이터가 아니라 **train 데이터에만** 적용했다는 것입니다. 모형은 train 데이터가 가진 패턴만 가지고 학습되어야 하며, test 데이터는 어디까지나 "모형이 한 번도 보지 못한 새로운 데이터에서도 잘 맞추는지"를 확인하기 위한 용도로 남겨둬야 합니다. summary 결과를 보면 5개 변수 모두 p-value가 0.05보다 작아 통계적으로 유의합니다.

**셀 4) 오즈비 계산 (문제 2-1 정답)**

In [10]:
import numpy as np

odds_ratio = np.exp(model.params['num_late_payments'])
print("2-1 정답:", round(odds_ratio, 3))

2-1 정답: 1.562


로지스틱 회귀에서 각 변수의 계수(coef)에 지수함수(np.exp())를 취하면 오즈비(Odds Ratio)로 해석할 수 있습니다. 계수 자체는 로그 오즈(log-odds) 단위라 직관적으로 와닿지 않지만, 지수를 취하면 "몇 배"라는 이해하기 쉬운 형태로 바뀝니다. num_late_payments의 계수는 0.4463이었고, 여기에 지수를 취한 값이 약 1.562입니다.
이 오즈비는 이진(범주형) 변수가 아니라 연속형 변수에 대한 것이라는 점에 주의해야 합니다. 오즈비는 항상 "그 변수가 1단위 증가할 때, 오즈가 몇 배가 되는가"를 의미하는데, 범주형 변수(0/1)에서는 이 1단위 증가가 곧 "값이 0에서 1로 바뀌는 것"(예: no에서 yes로)을 뜻하고, 연속형 변수에서는 문자 그대로 "숫자가 1만큼 늘어나는 것"을 뜻합니다. 여기서는 num_late_payments가 정수(연체 횟수)이므로, 오즈비 1.562는 "연체 횟수가 1회 더 많아질 때마다 연체 오즈가 약 1.562배씩 증가한다"는 의미가 됩니다.

**셀 5) test 데이터의 연체 확률 예측**

In [11]:
X_test = test.drop(columns=['id', 'delinquent'])
y_test = test['delinquent']
X_test = sm.add_constant(X_test, has_constant='add')

pred_prob = model.predict(X_test)
print(pred_prob.head())

0    0.294007
1    0.191346
2    0.352144
3    0.143615
4    0.247304
dtype: float64


test 데이터에도 train과 동일하게 상수항 컬럼을 추가해야 모형이 사용한 컬럼 구조와 맞아떨어집니다. 여기서 `sm.add_constant(X_test, has_constant='add')`처럼 `has_constant='add'`라는 옵션을 추가로 사용했는데, 그 이유가 있습니다. `sm.add_constant()`는 기본적으로 "이미 모든 값이 1인 컬럼이 있는지" 확인해서, 데이터 안에 우연히 그런 컬럼이 있으면 새로 추가하지 않고 건너뛰는 경우가 있습니다. `has_constant='add'`는 이런 경우까지 고려해서 "무조건 상수항 컬럼을 추가하라"고 명시적으로 지정하는 옵션으로, 특히 test 데이터처럼 행 개수가 적거나 특정 값들이 우연히 겹칠 수 있는 상황에서 안전하게 사용할 수 있습니다.

`model.predict(X_test)`는 train 데이터로 학습된 모형에 test 데이터를 넣어 각 고객의 연체 확률을 계산합니다. 이 모형은 train 데이터만 보고 학습되었을 뿐 test 데이터의 실제 정답(y_test)은 전혀 사용하지 않았다는 점을 다시 한번 기억해두면 좋습니다.

**셀 6) 임계값 0.3 기준으로 분류**

In [12]:
pred_class = (pred_prob > 0.3).astype(int)
print(pred_class.head())
print(pred_class.value_counts())

0    0
1    0
2    1
3    0
4    0
dtype: int64
0    236
1     64
Name: count, dtype: int64


`pred_prob > 0.3`은 각 확률이 0.3보다 큰지를 True/False로 판별하고, `.astype(int)`로 이를 1/0 정수로 바꿨습니다. 이렇게 만들어진 pred_class가 곧 "이 임계값 기준으로 모형이 내린 최종 판단"입니다. 여기서 왜 하필 0.5가 아니라 0.3을 기준으로 삼았는지 생각해볼 필요가 있습니다. 일반적으로 임계값 0.5는 "확률이 반을 넘으면 그쪽으로 판단한다"는 가장 무난한 기준이지만, 연체처럼 실제 발생 비율이 낮은(이 데이터에서는 약 18~23%) 상황에서는 0.5를 기준으로 삼으면 모형이 지나치게 보수적으로 "정상"이라고만 판단해서 실제 연체자를 거의 잡아내지 못할 수 있습니다. 그래서 이런 경우 임계값을 낮춰서(여기서는 0.3) "조금이라도 위험 신호가 보이면 연체 가능성이 있다고 판단"하도록 기준을 조정하는 것이며, 문제에서 이 임계값을 직접 지정해준 것도 이런 맥락과 관련이 있습니다.

**셀 7) 혼동행렬 구성 요소 계산 및 민감도 (문제 2-2 정답)**

In [13]:
TP = ((pred_class == 1) & (y_test == 1)).sum()
FN = ((pred_class == 0) & (y_test == 1)).sum()
TN = ((pred_class == 0) & (y_test == 0)).sum()
FP = ((pred_class == 1) & (y_test == 0)).sum()
print("TP:", TP, "FN:", FN, "TN:", TN, "FP:", FP)

sensitivity = TP / (TP + FN)
print("2-2 정답:", round(sensitivity, 3))

TP: 25 FN: 29 TN: 207 FP: 39
2-2 정답: 0.463


혼동행렬은 예측(pred_class)과 실제값(y_test)의 조합을 4가지 경우로 나눈 표입니다.
- TP(True Positive): 실제로 연체(1)인데 모형도 연체(1)라고 맞춘 경우
- FN(False Negative): 실제로 연체(1)인데 모형은 정상(0)이라고 놓친 경우
- TN(True Negative): 실제로 정상(0)인데 모형도 정상(0)이라고 맞춘 경우
- FP(False Positive): 실제로 정상(0)인데 모형은 연체(1)라고 잘못 판단한 경우

`(pred_class == 1) & (y_test == 1)`처럼 두 조건을 `&`로 동시에 만족하는 행만 True가 되도록 만든 뒤 `.sum()`으로 그 개수를 셌습니다. (True는 1, False는 0으로 취급되므로 합계가 곧 개수가 됩니다.)

민감도(Sensitivity)는 `TP / (TP + FN)`으로 계산하며, 이는 "실제로 연체인 사람들(TP+FN) 중에서 모형이 연체라고 제대로 잡아낸 비율"을 의미합니다. 이 값이 약 0.463이라는 것은, 실제 연체자 100명 중 약 46명 정도만 모형이 미리 잡아낼 수 있었다는 뜻입니다. 민감도는 "위험한 대상을 놓치지 않는 능력"을 나타내는 지표이므로, 질병 진단이나 이번처럼 연체 위험 감지처럼 "놓치면 손실이 큰" 상황에서 특히 중요하게 다뤄집니다.

**셀 8) 특이도 계산 (문제 2-3 정답)**

In [14]:
specificity = TN / (TN + FP)
print("2-3 정답:", round(specificity, 3))

2-3 정답: 0.841


특이도(Specificity)는 `TN / (TN + FP)`로 계산하며, "실제로 정상인 사람들(TN+FP) 중에서 모형이 정상이라고 제대로 맞춘 비율"을 의미합니다. 셀 7의 TN, FP는 이미 계산해뒀으므로 여기서는 나누기만 하면 됩니다. 특이도가 약 0.841이라는 것은, 실제 정상 고객 100명 중 약 84명은 모형이 정확히 정상으로 분류했다는 뜻입니다.

민감도(0.463)와 특이도(0.841)를 함께 보면, 이 모형은 "정상인 사람을 정상으로 판단하는 데는 상당히 정확하지만, 실제 연체자를 잡아내는 능력은 상대적으로 약하다"는 특징을 가지고 있음을 알 수 있습니다. 만약 연체자를 더 적극적으로 잡아내고 싶다면 임계값을 0.3보다 더 낮춰볼 수 있는데, 다만 그렇게 하면 민감도는 올라가는 대신 특이도는 낮아지는(정상 고객을 연체로 오판하는 경우가 늘어나는) 트레이드오프가 발생합니다.

### 전체 코드

In [15]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_csv('card_delinquency.csv')

train = df[df['id'] <= 1200].reset_index(drop=True)
test = df[df['id'] > 1200].reset_index(drop=True)

# train으로 모형 적합
X_train = train.drop(columns=['id', 'delinquent'])
y_train = train['delinquent']
X_train = sm.add_constant(X_train)
model = sm.Logit(y_train, X_train).fit()

# 2-1: 오즈비
odds_ratio = round(np.exp(model.params['num_late_payments']), 3)
print("2-1 정답:", odds_ratio)

# test로 확률 예측 및 임계값 0.3 기준 분류
X_test = test.drop(columns=['id', 'delinquent'])
y_test = test['delinquent']
X_test = sm.add_constant(X_test, has_constant='add')
pred_prob = model.predict(X_test)
pred_class = (pred_prob > 0.3).astype(int)

# 혼동행렬 구성 요소
TP = ((pred_class == 1) & (y_test == 1)).sum()
FN = ((pred_class == 0) & (y_test == 1)).sum()
TN = ((pred_class == 0) & (y_test == 0)).sum()
FP = ((pred_class == 1) & (y_test == 0)).sum()

# 2-2: 민감도, 2-3: 특이도
sensitivity = round(TP / (TP + FN), 3)
specificity = round(TN / (TN + FP), 3)
print("2-2 정답:", sensitivity)
print("2-3 정답:", specificity)

Optimization terminated successfully.
         Current function value: 0.492935
         Iterations 6
2-1 정답: 1.562
2-2 정답: 0.463
2-3 정답: 0.841


### 정리

이 문제는 로지스틱 회귀를 "학습"과 "평가"로 명확히 분리해서 다뤘습니다. train으로만 모형을 적합하고, test로는 오직 예측과 평가만 수행했다는 원칙이 핵심입니다. 또한 확률을 0.3이라는 임계값으로 분류한 뒤, TP·FN·TN·FP라는 네 가지 경우의 수를 직접 세어 민감도와 특이도를 계산하는 과정을 다뤘습니다. 개인적으로는, 이 혼동행렬 관련 지표들은 공식을 외우기보다 "지금 내가 구하는 게 실제 양성 중 맞춘 비율인지(민감도), 실제 음성 중 맞춘 비율인지(특이도)"를 말로 먼저 떠올려보면 헷갈리지 않고 공식을 스스로 유도할 수 있어서 훨씬 안전합니다.

---

## 이번 회차에서 사용한 핵심 개념 정리
| 상황 | 함수/개념 | 핵심 이유 |
|---|---|---|
| 두 범주형 변수의 교차표 만들기 | `pd.crosstab(a, b)` | 두 조건을 동시에 만족하는 건수를 표로 집계 |
| 카이제곱 독립성검정 | `chi2_contingency(교차표)` | 검정통계량, p-value, 자유도, 기대빈도를 한 번에 반환 |
| 관측값과 이론적 기댓값의 차이를 하나의 지표로 | 카이제곱 통계량 | 값이 클수록 독립 가정과 실제 데이터의 괴리가 큼 |
| p-value로 가설 판단 | `p < 유의수준` 이면 기각 | p-value는 "귀무가설이 맞다면 이런 결과가 나올 확률" |
| id처럼 순번 기준으로 train/test 나누기 | `df[df['id'] <= 기준]` / `df[df['id'] > 기준]` | 조건으로 행을 분할, `reset_index(drop=True)`로 인덱스 정리 |
| test 데이터에도 상수항 추가 시 안전장치 | `sm.add_constant(X, has_constant='add')` | 우연히 상수 컬럼처럼 보여도 무조건 추가되도록 강제 |
| 확률을 임계값 기준으로 분류 | `(확률 > 임계값).astype(int)` | 임계값은 상황(클래스 불균형 등)에 따라 조정 가능 |
| 혼동행렬 구성 요소 세기 | `((조건1) & (조건2)).sum()` | True/False가 1/0으로 취급되어 합계가 개수가 됨 |
| 민감도 / 특이도 | `TP/(TP+FN)` / `TN/(TN+FP)` | 실제 양성을 잘 잡아내는지 / 실제 음성을 잘 걸러내는지 |